# 11 - Eval Generation Grid (plan_eval.md Phase E1)

Builds the two evaluation corpora consumed by every later eval notebook:

| corpus | contents | rows | file |
|---|---|---|---|
| main grid | 200 C4 prompts x {plain, l1_only, l2_only, dual} x 200 tokens | 800 | `eval_results/generation/gen_main.csv` |
| delta sweep | first 50 of those prompts x delta_public {0,1,2,3,4}, dual only | 250 | `eval_results/generation/gen_sweep.csv` |
| sweep PPL | external perplexity (Qwen1.5-7B, 4-bit) of 200 random sweep texts + 50 plain baselines | 250 | `eval_results/generation/ppl_sweep.csv` |

Conventions follow `11_qualitymatrix.ipynb`: OPT loaded fp16 on GPU with topic-routing
tensors recast fp32; per-row seeds; chunked Drive checkpointing with id-resume.
Phased because OPT and the Qwen judge cannot share 16 GB VRAM.

**Honest T4 budget**: ~1050 generations at ~9-13 s each -> main grid 1.8-2.7 h,
sweep 35-55 min, judge download+scoring ~30 min => **2.5-3.5 h total** (A100/L4: ~1-1.5 h).
A disconnect loses at most one unflushed chunk.

In [1]:
!git clone https://github.com/pravaspaudel/Dual_watermarking_Scheme.git
%cd Dual_watermarking_Scheme
!pip install -q transformers accelerate bitsandbytes scipy pandas matplotlib tqdm

Cloning into 'Dual_watermarking_Scheme'...


remote: Enumerating objects: 272, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 272 (delta 63), reused 76 (delta 24), pack-reused 151 (from 1)
Receiving objects: 100% (272/272), 43.78 MiB | 19.77 MiB/s, done.
Resolving deltas: 100% (122/122), done.
/content/Dual_watermarking_Scheme
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.1 MB/s eta 0:00:00:00:0100:01


In [3]:
import gc
import os
import time

import numpy as np
import pandas as pd
import torch

from google.colab import drive
drive.mount('/content/drive')

from tqdm.auto import tqdm
from src.utils.model import load_model
from src.utils.key_manager import generate_key
from src.watermark.dual_layer import DualWaterMarking
from src.metrics.perplexity import (
    load_judge_model, split_continuation, compute_external_perplexity,
)

# ----------------------------- CONFIG ---------------------------------------
CONFIG = {
    "N_PROMPTS"        : 200,     # main-grid prompts (random_state=0 sample)
    "SWEEP_N"          : 50,      # first K of those prompts reused in the sweep
    "DELTA_SWEEP"      : [0, 1, 2, 3, 4],
    "DELTA_PUBLIC_OP"  : 2.0,     # operating point until E2 freezes a better one
    "DELTA_PRIVATE"    : 0.7,
    "MAX_NEW_TOKENS"   : 200,
    "SEED_BASE"        : 1000,    # per-row seed = SEED_BASE + i
    "CHUNK"            : 25,      # flush cadence (rows buffered before writing)
    "VARIANTS"         : ["plain", "l1_only", "l2_only", "dual"],
    "JUDGE_NAME"       : "Qwen/Qwen1.5-7B",
    "LOAD_4BIT"        : True,    # REQUIRED on T4: fp16 7B ~15.4GB > 15GB VRAM
    "PPL_PER_DELTA"    : 40,      # random sweep texts scored per delta (40x5=200)
    "PPL_BASELINES"    : 50,      # plain-output baselines scored alongside
    "SAMPLE_SEED"      : 0,
}

DRIVE  = "/content/drive/MyDrive/minor_project"
GENDIR = f"{DRIVE}/eval_results/generation"
FIGDIR = f"{DRIVE}/eval_results/figures"
os.makedirs(GENDIR, exist_ok=True)
os.makedirs(FIGDIR, exist_ok=True)

MAIN_CSV  = f"{GENDIR}/gen_main.csv"
SWEEP_CSV = f"{GENDIR}/gen_sweep.csv"
PPL_CSV   = f"{GENDIR}/ppl_sweep.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Runtime > Change runtime type > T4 GPU"

def merge_and_save(buf, path):
    old = pd.read_csv(path) if os.path.exists(path) else None
    merged = pd.concat([old, pd.DataFrame(buf)]) if old is not None else pd.DataFrame(buf)
    merged.to_csv(path, index=False)

print("device:", DEVICE)

Mounted at /content/drive
device: cuda


## Phase A1 - main grid (800 generations)

OPT in fp16 (repo convention from `11_qualitymatrix.ipynb` cell 20: routing tensors are
recast to fp32 afterwards so cosine routing stays identical to previous runs).
Each prompt generates all four variants via `wm.watermark(..., include_single_layers=True)`
and is reshaped to the long-format schema from PLAN.md Phase 0.

In [5]:
model, tokenizer, VOCAB_SIZE = load_model("facebook/opt-2.7b")
model.half().to(DEVICE).eval()         

key = os.getenv("WATERMARK_SECRET_KEY") or generate_key()

wm = DualWaterMarking(
    model, tokenizer, key=key,
    greenlist_dir="data/greenlist", split="all",
    green_fraction=0.5, prev_token_size=5,
    delta_public=CONFIG["DELTA_PUBLIC_OP"],
    delta_private=CONFIG["DELTA_PRIVATE"],
    max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
)

# fp32 routing geometry on top of the fp16 model (qualitymatrix convention)
wm.topic_matrix = wm.topic_matrix.to(device=DEVICE, dtype=torch.float32)
wm.normed_embeddings = wm.normed_embeddings.to(device=DEVICE, dtype=torch.float32)

print("key fingerprint:", key[:8], "| topics:", wm.topics)


def long_rows(i, prompt, result):
    """one wm.watermark() result dict -> 4 long-format records"""
    mapping = {
        "plain":     "plain_output",
        "l1_only":   "layer1_only_output",
        "l2_only":   "layer2_only_output",
        "dual":      "dual_watermarked_output",
    }
    rows = []
    for variant, col in mapping.items():
        text = result[col]
        rows.append({
            "id": i, "prompt_text": prompt, "topic": result["topic"],
            "variant": variant,
            "delta_public": CONFIG["DELTA_PUBLIC_OP"],
            "delta_private": CONFIG["DELTA_PRIVATE"],
            "seed": CONFIG["SEED_BASE"] + i,
            "gen_tokens": len(tokenizer.encode(text)),
            "text": text,
        })
    return rows

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

model and tokenizer of facebook/opt-2.7b loaded with vocab_size 50265
key fingerprint: 1fa8742a | topics: ['entertainment', 'finance', 'history', 'medicine', 'politics', 'science', 'sports', 'technology']


In [6]:
c4_df = pd.read_csv("data/extracted/c4_samples.csv")
sampled = c4_df.sample(n=CONFIG["N_PROMPTS"], random_state=CONFIG["SAMPLE_SEED"])
print(f"sampling {len(sampled)} of {len(c4_df)} C4 rows "
      f"(original index kept as id: {list(sampled.index[:5])}...)")

done_ids = set()
if os.path.exists(MAIN_CSV):
    done_ids = set(pd.read_csv(MAIN_CSV)["id"].unique())
    print(f"resume: {len(done_ids)} ids already generated")

buf, t_main0 = [], time.time()
for i, prompt in enumerate(tqdm(sampled["prompt_text"].tolist(), desc="main grid")):
    if i in done_ids:
        continue
    torch.manual_seed(CONFIG["SEED_BASE"] + i)
    buf.extend(long_rows(i, prompt, wm.watermark([prompt], include_single_layers=True).iloc[0]))
    if len(buf) >= CONFIG["CHUNK"] * len(CONFIG["VARIANTS"]):
        merge_and_save(buf, MAIN_CSV)
        buf = []
if buf:
    merge_and_save(buf, MAIN_CSV)

main_df = pd.read_csv(MAIN_CSV)
t_main = time.time() - t_main0
print(f"main grid: {len(main_df)} rows ({main_df['id'].nunique()} prompts) "
      f"in {t_main/60:.1f} min") 

sampling 200 of 1000 C4 rows (original index kept as id: [993, 859, 298, 553, 672]...)


main grid:   0%|          | 0/200 [00:00<?, ?it/s]

main grid: 800 rows (200 prompts) in 77.8 min


## Phase A2 - delta_public sweep (dual only, 250 generations)

First 50 sampled prompts, regenerated as pure-dual at each sweep delta.
`torch.manual_seed(SEED_BASE + i)` is reset per (prompt, delta) so deltas are compared
on the same sampling stream.

In [7]:
sweep_prompts = sampled["prompt_text"].tolist()[:CONFIG["SWEEP_N"]]

done_sweep = set()
if os.path.exists(SWEEP_CSV):
    done_sweep = set(map(tuple, pd.read_csv(SWEEP_CSV)[["id", "delta_public_sweep"]].values))
    print(f"resume: {len(done_sweep)} (id, delta) pairs already generated")

buf, t_sweep0 = [], time.time()
for d in CONFIG["DELTA_SWEEP"]:
    wm.delta_public = float(d)                    
    for i, prompt in enumerate(tqdm(sweep_prompts, desc=f"delta={d}")):
        if (i, float(d)) in done_sweep:
            continue
        torch.manual_seed(CONFIG["SEED_BASE"] + i)
        topic, ranked = wm.extract_topic(prompt)
        inputs = wm._inputs(prompt)
        text = wm._generate(inputs, wm._make_processor(topic, 1.0, 1.0))
        buf.append({"id": i, "prompt_text": prompt, "topic": topic,
                    "delta_public_sweep": float(d),
                    "seed": CONFIG["SEED_BASE"] + i,
                    "gen_tokens": len(tokenizer.encode(text)), "text": text})
        if len(buf) >= CONFIG["CHUNK"]:
            merge_and_save(buf, SWEEP_CSV)
            buf = []
if buf:
    merge_and_save(buf, SWEEP_CSV)
wm.delta_public = CONFIG["DELTA_PUBLIC_OP"]        
sweep_df = pd.read_csv(SWEEP_CSV)
t_sweep = time.time() - t_sweep0
print(f"sweep: {len(sweep_df)} rows in {t_sweep/60:.1f} min")
assert len(sweep_df) == CONFIG["SWEEP_N"] * len(CONFIG["DELTA_SWEEP"])

delta=0:   0%|          | 0/50 [00:00<?, ?it/s]

delta=1:   0%|          | 0/50 [00:00<?, ?it/s]

delta=2:   0%|          | 0/50 [00:00<?, ?it/s]

delta=3:   0%|          | 0/50 [00:00<?, ?it/s]

delta=4:   0%|          | 0/50 [00:00<?, ?it/s]

sweep: 250 rows in 27.1 min


### QA - main grid + sweep

In [ ]:
# row counts
assert len(main_df) == CONFIG["N_PROMPTS"] * len(CONFIG["VARIANTS"]), "main grid incomplete"
assert main_df.groupby("variant")["id"].nunique().eq(CONFIG["N_PROMPTS"]).all(), "variant gap"
assert not main_df["text"].isna().any() and (main_df["text"].str.len() > 0).all(), "empty text"

# token-length histograms per variant
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, v in zip(axes, CONFIG["VARIANTS"]):
    sub = main_df[main_df["variant"] == v]
    ax.hist(sub["gen_tokens"], bins=25, color="#1f77b4", alpha=0.85)
    ax.axvline(sub["gen_tokens"].median(), color="k", ls="--", lw=1)
    ax.set_title(f"{v}\nmed {int(sub['gen_tokens'].median())}")
    ax.set_xlabel("generated tokens")
axes[0].set_ylabel("count")
fig.suptitle("Generation lengths per variant (main grid)")
fig.tight_layout()
fig.savefig(f"{FIGDIR}/e1_gen_length_histograms.png", dpi=300)
plt.show()

ex = main_df[(main_df["id"] == 0)].set_index("variant")
print("PROMPT:", ex.iloc[0]["prompt_text"], "\n")
for v in CONFIG["VARIANTS"]:
    print(f"--- {v}: {ex.loc[v]['text'][:180]}...\n")
print("QA passed.")

## Phase B - free OPT, load Qwen1.5-7B judge (4-bit)

The judge comes from `src/metrics/perplexity.py`; 4-bit is mandatory on a T4
(fp16 weights alone are ~15.4 GB). Requires the pushed version of
`load_judge_model(torch_dtype=..., load_in_4bit=...)`.

In [ ]:
wm, model, tokenizer = None, None, None
del wm, model, tokenizer
gc.collect(); torch.cuda.empty_cache()

judge_model, judge_tokenizer = load_judge_model(
    CONFIG["JUDGE_NAME"], device=DEVICE,
    torch_dtype=torch.float16, load_in_4bit=CONFIG["LOAD_4BIT"],
)
print(f"judge ready: {CONFIG['JUDGE_NAME']} | 4-bit={CONFIG['LOAD_4BIT']} "
      f"| VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Phase C - sweep perplexity (Qwen1.5-7B)

Random 200 of the 250 sweep texts (40 per delta, seeded draw) get external PPL,
plus all 50 plain baselines from the main grid for a reference line ->
the quality-vs-strength curve for the report (feeds E2's frozen-delta decision).

In [ ]:
rng = np.random.default_rng(CONFIG["SAMPLE_SEED"])
picked = []
for d in CONFIG["DELTA_SWEEP"]:
    pool = sweep_df[sweep_df["delta_public_sweep"] == d]
    take = pool.sample(n=min(CONFIG["PPL_PER_DELTA"], len(pool)),
                       random_state=int(rng.integers(1 << 31)))
    picked.append(take)
ppl_targets = pd.concat(picked)

base_ids = sorted(main_df["id"].unique())[:CONFIG["PPL_BASELINES"]]
baselines = main_df[(main_df["variant"] == "plain") & (main_df["id"].isin(base_ids))]
baselines = baselines.assign(delta_public_sweep=-1.0)   # sentinel = baseline row

jobs = pd.concat([
    ppl_targets,
    baselines[["id", "prompt_text", "topic", "delta_public_sweep", "text"]],
])[["id", "prompt_text", "topic", "delta_public_sweep", "text"]]
print(f"PPL jobs: {len(jobs)} "
      f"({len(ppl_targets)} sweep + {len(baselines)} baselines)")

done_ppl = set()
if os.path.exists(PPL_CSV):
    done_ppl = set(map(tuple,
        pd.read_csv(PPL_CSV)[["id", "delta_public_sweep"]].values.astype(float)))
    print(f"resume: {len(done_ppl)} jobs already scored")

buf, t_ppl0 = [], time.time()
for rec in tqdm(jobs.to_dict("records"), desc="external PPL"):
    key_t = (float(rec["id"]), float(rec["delta_public_sweep"]))
    if key_t in done_ppl:
        continue
    cont = split_continuation(rec["prompt_text"], rec["text"])
    row = {"id": rec["id"], "delta_public_sweep": rec["delta_public_sweep"],
           "topic": rec["topic"],
           "variant": ("dual_sweep" if rec["delta_public_sweep"] >= 0 else "plain_baseline")}
    row["external_ppl"] = compute_external_perplexity(
        rec["prompt_text"], cont, judge_model, judge_tokenizer, DEVICE)
    buf.append(row)
    if len(buf) >= 5:
        merge_and_save(buf, PPL_CSV)
        buf = []
if buf:
    merge_and_save(buf, PPL_CSV)

ppl_df = pd.read_csv(PPL_CSV)
t_ppl = time.time() - t_ppl0
print(f"PPL scoring: {len(ppl_df)} rows in {t_ppl/60:.1f} min")

## Results - quality vs watermark strength

In [ ]:
curve = (ppl_df[ppl_df["delta_public_sweep"] >= 0]
         .groupby("delta_public_sweep")["external_ppl"]
         .agg(["mean", "std", "count"]).round(3))
base_mask = ppl_df["delta_public_sweep"] < 0
baseline_mean = ppl_df.loc[base_mask, "external_ppl"].mean()
baseline_std = ppl_df.loc[base_mask, "external_ppl"].std()
print(curve.to_string())
print(f"\nplain baseline: {baseline_mean:.3f} +/- {baseline_std:.3f} "
      f"(n={int(base_mask.sum())})")

plt.figure(figsize=(7, 4))
plt.errorbar(curve.index, curve["mean"], yerr=curve["std"],
             marker="o", capsize=4, color="#2ca02c", label="dual (watermarked)")
plt.axhline(baseline_mean, ls="--", color="#8c8c8c",
            label=f"plain baseline {baseline_mean:.2f}")
plt.xlabel("delta_public")
plt.ylabel("external PPL (Qwen1.5-7B, continuation-only)")
plt.title("Quality vs watermark strength - sweep subset")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGDIR}/e1_ppl_vs_delta.png", dpi=300)
plt.show()

print("\ntiming:")
print(f"  main grid : {t_main/60:6.1f} min")
print(f"  sweep gen : {t_sweep/60:6.1f} min")
print(f"  PPL score : {t_ppl/60:6.1f} min")